# **SETUP**

## **SETUP: Check the GPU**

In [ ]:
# SETUP — Cell 1: Check GPU

import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is available. Select a GPU runtime before continuing."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

## **SETUP: GPU details**

In [ ]:
# SETUP — Cell 2: GPU details

!nvidia-smi

## **SETUP: Install experiment dependencies**

In [ ]:
# SETUP — Cell 3: Install experiment dependencies

!pip install -q \
    transformers==4.57.6 \
    peft==0.20.0 \
    bitsandbytes==0.50.2 \
    accelerate \
    datasets \
    pandas

## **SETUP: Verify experiment environment**

In [ ]:
import torch
import transformers
import peft
import bitsandbytes as bnb

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

## **SETUP: Clone repository**


In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/llm-finetuning-for-implicatures")

if REPO_DIR.exists():
    raise RuntimeError(
        f"{REPO_DIR} already exists. Use a fresh Colab runtime rather than overwriting it."
    )

subprocess.run(
    [
        "git",
        "clone",
        "https://github.com/RadwanRasul/llm-finetuning-for-implicatures.git",
        str(REPO_DIR),
    ],
    check=True,
)

print("\nRepository revision:")
subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
    check=True,
)

print("\nWorking-tree status:")
subprocess.run(
    ["git", "-C", str(REPO_DIR), "status", "--short"],
    check=True,
)


## **SETUP: Select model and verify Hugging Face access**

In [ ]:
# SETUP — Select model and verify Hugging Face access

from google.colab import userdata
from huggingface_hub import HfApi

# ============================================================
# CHANGE ONLY THIS VALUE WHEN SWITCHING MODELS
# Options: "270m", "1b", "4b"
# ============================================================
MODEL_SIZE = "270m"

MODEL_CONFIGS = {
    "270m": {
        "checkpoint": "google/gemma-3-270m-it",
        "label": "270m",
    },
    "1b": {
        "checkpoint": "google/gemma-3-1b-it",
        "label": "1b",
    },
    "4b": {
        "checkpoint": "google/gemma-3-4b-it",
        "label": "4b",
    },
}

if MODEL_SIZE not in MODEL_CONFIGS:
    raise ValueError(
        f"MODEL_SIZE must be one of {list(MODEL_CONFIGS)}, got {MODEL_SIZE!r}"
    )

MODEL_CHECKPOINT = MODEL_CONFIGS[MODEL_SIZE]["checkpoint"]
MODEL_LABEL = MODEL_CONFIGS[MODEL_SIZE]["label"]

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError("HF_TOKEN was not found in Colab Secrets.")

api = HfApi(token=hf_token)
info = api.model_info(MODEL_CHECKPOINT)

print("✓ Hugging Face access confirmed")
print("Model size:", MODEL_SIZE)
print("Model:", info.id)
print("Revision:", info.sha)

# **HELPERS**

## **HELPERS: Paths and environment**

In [ ]:
from google.colab import userdata
from pathlib import Path
import os

PROJECT_DIR = Path("/content/llm-finetuning-for-implicatures")

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError("HF_TOKEN was not found in Colab Secrets.")

os.environ["HF_TOKEN"] = hf_token
os.environ["PYTHONPATH"] = str(PROJECT_DIR)


def training_dir(fold, condition):
    return (
        PROJECT_DIR
        / "results"
        / "training"
        / MODEL_SIZE
        / condition
        / f"fold_{fold}"
    )


def evaluation_dir(fold, condition):
    return (
        PROJECT_DIR
        / "results"
        / "evaluation"
        / MODEL_SIZE
        / condition
        / f"fold_{fold}"
    )


def output_name(fold, condition):
    return f"gemma3_{MODEL_LABEL}_fold{fold}_{condition}"

## **HELPERS: Verification + packaging**

In [ ]:
# HELPERS — Cell 8: Verification and packaging

import shutil
import pandas as pd


def verify_predictions(directory):
    predictions_path = directory / "predictions.csv"
    config_path = directory / "run_config.json"

    if not predictions_path.is_file():
        raise RuntimeError(f"Missing predictions: {predictions_path}")

    if not config_path.is_file():
        raise RuntimeError(f"Missing run config: {config_path}")

    predictions = pd.read_csv(predictions_path)

    if len(predictions) != 160:
        raise RuntimeError(
            f"Expected 160 predictions, found {len(predictions)}."
        )

    counts = predictions["evaluation_setting"].value_counts().to_dict()

    if counts.get("in_situation") != 80:
        raise RuntimeError(
            f"Expected 80 in-situation predictions, "
            f"found {counts.get('in_situation', 0)}."
        )

    if counts.get("cross_situation") != 80:
        raise RuntimeError(
            f"Expected 80 cross-situation predictions, "
            f"found {counts.get('cross_situation', 0)}."
        )


def make_zip(source_dir, name):
    zip_base = Path("/content") / name

    return shutil.make_archive(
        str(zip_base),
        "zip",
        root_dir=source_dir,
    )

## **HELPERS: Run functions**

In [ ]:
import subprocess
import shutil


def run_base(fold):
    condition = "base"
    eval_dir = evaluation_dir(fold, condition)

    subprocess.run(
        [
            "python",
            "scripts/evaluate_gemma3.py",
            "--model-size", MODEL_SIZE,
            "--fold", str(fold),
            "--condition", condition,
        ],
        cwd=PROJECT_DIR,
        check=True,
    )

    verify_predictions(eval_dir)

    zip_path = make_zip(
        eval_dir,
        output_name(fold, condition),
    )

    print()
    print(f"✓ Fold {fold} Base evaluation complete")
    print("✓ 160 predictions verified")
    print("✓ Evaluation outputs packaged")
    print("Download:")
    print(zip_path)


def run_finetuned(fold, condition):
    if condition not in {"no_hint", "hint"}:
        raise ValueError(
            "condition must be 'no_hint' or 'hint'"
        )

    train_dir = training_dir(fold, condition)
    eval_dir = evaluation_dir(fold, condition)

    subprocess.run(
        [
            "python",
            "scripts/train_gemma3.py",
            "--model-size", MODEL_SIZE,
            "--fold", str(fold),
            "--condition", condition,
        ],
        cwd=PROJECT_DIR,
        check=True,
    )

    required_training_outputs = [
        train_dir / "adapter",
        train_dir / "run_config.json",
        train_dir / "training_history.csv",
    ]

    for path in required_training_outputs:
        if not path.exists():
            raise RuntimeError(
                f"Missing training output: {path}"
            )

    subprocess.run(
        [
            "python",
            "scripts/evaluate_gemma3.py",
            "--model-size", MODEL_SIZE,
            "--fold", str(fold),
            "--condition", condition,
        ],
        cwd=PROJECT_DIR,
        check=True,
    )

    verify_predictions(eval_dir)

    name = output_name(fold, condition)
    bundle_dir = Path("/content") / f"{name}_bundle"

    if bundle_dir.exists():
        shutil.rmtree(bundle_dir)

    shutil.copytree(
        train_dir,
        bundle_dir / "training",
    )

    shutil.copytree(
        eval_dir,
        bundle_dir / "evaluation",
    )

    zip_path = make_zip(
        bundle_dir,
        name,
    )

    print()
    print(f"✓ Fold {fold} {condition} training complete")
    print("✓ Training outputs verified")
    print(f"✓ Fold {fold} {condition} evaluation complete")
    print("✓ 160 predictions verified")
    print("✓ Training + evaluation outputs packaged")
    print("Download:")
    print(zip_path)

# **EXPERIMENTS**

## **FOLD 1**

### **BASE**

In [ ]:
run_base(fold=1)

### **NO-HINT**

In [ ]:
run_finetuned(fold=1, condition="no_hint")

### **HINT**

In [ ]:
run_finetuned(fold=1, condition="hint")

## **FOLD 2**

### **BASE**

In [ ]:
run_base(fold=2)

### **NO-HINT**

In [ ]:
run_finetuned(fold=2, condition="no_hint")

### **HINT**

In [ ]:
run_finetuned(fold=2, condition="hint")

## **FOLD 3**

### **BASE**

In [ ]:
run_base(fold=3)

### **NO-HINT**

In [ ]:
run_finetuned(fold=3, condition="no_hint")

### **HINT**

In [ ]:
run_finetuned(fold=3, condition="hint")

## **FOLD 4**

### **BASE**

In [ ]:
run_base(fold=4)

### **NO-HINT**

In [ ]:
run_finetuned(fold=4, condition="no_hint")

### **HINT**

In [ ]:
run_finetuned(fold=4, condition="hint")

## **FOLD 5**

### **BASE**

In [ ]:
run_base(fold=5)

### **NO-HINT**

In [ ]:
run_finetuned(fold=5, condition="no_hint")

### **HINT**

In [ ]:
run_finetuned(fold=5, condition="hint")